# 03｜主项目：读取真实H&E与HER2-IHC配对数据

对应第4章与第5章。先确认来源、配对、尺度与集合划分，再进入模型训练。本笔记本不会下载不明镜像，也不会用生成图替代缺失的真实IHC。

版本：KYDW 2.0。按从上到下顺序运行。代码保留完整实现；涉及外部数据时，请先按学生手册取得数据并遵守来源许可。测试输出由实际运行生成，本文件未预填模型性能。

## KYDW 新成员培训与考核 · 运行入口

先按课程第5章取得 BCI 的 H&E / HER2-IHC 配对数据并添加为 Input，再检查 BCI_ROOT、LAYOUT 与 HE_SIDE。当前版本不附带 BCI 数据。

[活动主页](https://lhy1007.github.io/KYDW_TRY/programs/member-training/index.html) · [本节说明与代码](https://lhy1007.github.io/KYDW_TRY/programs/member-training/lessons/05-data.html#practice-03)

这些 Notebook 提供课程代码。实验结果由你在实际数据上的运行产生。

## 公共工具区

下方代码包含数据读取、网络、训练和评价函数。第一次学习时先运行这一格，再继续后面的任务；对应章节会逐步解释用到的函数。展开本格可以查看全部实现，同一份源代码也位于项目包的 `src/kydw_course.py`。不要通过删改安全检查来绕过配对或划分错误。

In [ ]:
"""KYDW 2.0: 可检查的教学实现。外部数据须按原始许可取得。

CPU 可完成环境测试；真实图像训练建议在 Kaggle GPU 上执行。
此文件与 Notebook 中的工具代码由同一份源文件生成。
"""
from __future__ import annotations
import os, sys, json, math, random, hashlib, platform, time, zipfile, warnings
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, asdict
from typing import Optional
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from skimage.metrics import structural_similarity
from skimage.color import rgb2hed
from skimage.filters import threshold_otsu
from scipy.ndimage import gaussian_filter

VERSION = '2.0.0'
IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # 不强制所有 CUDA 算子使用确定性实现；仍保存环境供复查。


def device_for_training():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def work_root():
    p = Path(os.environ.get('KYDW_OUTPUT_ROOT', '/kaggle/working/kydw_v2' if Path('/kaggle').exists() else './kydw_outputs'))
    p.mkdir(parents=True, exist_ok=True)
    return p.resolve()


def safe_json(value):
    if isinstance(value, dict): return {str(k): safe_json(v) for k,v in value.items()}
    if isinstance(value, (list,tuple)): return [safe_json(v) for v in value]
    if isinstance(value, (np.bool_,)): return bool(value)
    if isinstance(value, (np.integer,)): return int(value)
    if isinstance(value, (np.floating,)): value=float(value)
    if isinstance(value,float) and not math.isfinite(value): return None
    if isinstance(value, Path): return str(value)
    return value


def write_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(safe_json(value), ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')


def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()


def environment_record():
    import torchvision, sklearn, skimage
    return dict(course_version=VERSION, utc=datetime.now(timezone.utc).isoformat(),
        python=sys.version, platform=platform.platform(), torch=str(torch.__version__),
        torchvision=str(torchvision.__version__), numpy=np.__version__, pandas=pd.__version__,
        sklearn=sklearn.__version__, skimage=skimage.__version__, cuda=torch.version.cuda,
        gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


def fresh_run(folder, overwrite=False):
    p=Path(folder)
    if p.exists() and any(p.iterdir()) and not overwrite:
        raise FileExistsError(f'{p} 已有结果。请更换 RUN_NAME，避免把两次实验混在一起。')
    p.mkdir(parents=True, exist_ok=True)
    write_json(p/'environment.json', environment_record())
    return p


def tensor_image(arr):
    a=np.asarray(arr, dtype=np.float32)
    if a.ndim==2: a=a[...,None]
    return torch.from_numpy(np.ascontiguousarray(a.transpose(2,0,1)))


def numpy_image(t):
    return t.detach().cpu().permute(1,2,0).numpy()


def read_rgb(path):
    with Image.open(path) as im:
        # 按文件中存储的坐标读入；不分别按 EXIF 旋转已配准图像。
        return im.convert('RGB').copy()


def finite_loss(loss):
    if not bool(torch.isfinite(loss).all()):
        raise FloatingPointError('损失出现 NaN/Inf；已停止，检查像素范围、标签与学习率。')


def loader(dataset, batch_size=8, shuffle=False, seed=42):
    g=torch.Generator().manual_seed(seed)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, generator=g, pin_memory=torch.cuda.is_available())


def save_curve(history, fields, output, ylabel):
    df=pd.DataFrame(history)
    fig,ax=plt.subplots(figsize=(7,4))
    for col in fields:
        if col in df: ax.plot(df['epoch'], df[col], label=col)
    ax.set(xlabel='Epoch', ylabel=ylabel); ax.legend(); fig.tight_layout()
    fig.savefig(output, dpi=180); plt.close(fig)


def show_strip(images, names, output=None):
    arrays=[]
    for a in images:
        if torch.is_tensor(a): a=numpy_image(a)
        a=np.asarray(a)
        if a.ndim==2: a=np.repeat(a[...,None],3,axis=-1)
        if a.shape[-1]==1: a=np.repeat(a,3,axis=-1)
        arrays.append(np.clip(a,0,1))
    if len({x.shape for x in arrays}) != 1: raise ValueError('对照图必须具有相同尺寸。')
    h,w,_=arrays[0].shape
    fig,ax=plt.subplots(figsize=(max(4,3*len(arrays)),3.5))
    ax.imshow(np.concatenate(arrays,axis=1))
    ax.set_xticks([(i+.5)*w for i in range(len(names))],names)
    ax.set_yticks([]); fig.tight_layout()
    if output: fig.savefig(output,dpi=180,bbox_inches='tight')
    plt.show(); plt.close(fig)


# ---- 网络：数字分类、像素分割、图像重建 ----
class DigitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.head=nn.Sequential(nn.Flatten(), nn.Linear(32*7*7,64),nn.ReLU(),nn.Linear(64,10))
    def forward(self,x): return self.head(self.features(x))

class LinearDigit(nn.Module):
    def __init__(self):
        super().__init__(); self.layer=nn.Linear(28*28,10)
    def forward(self,x): return self.layer(x.flatten(1))

class ConvBlock(nn.Module):
    def __init__(self,cin,cout):
        super().__init__()
        # GroupNorm 对小批量友好；这里是教学网络，与原始 U-Net 实现有区别。
        groups=4 if cout%4==0 else 1
        self.seq=nn.Sequential(nn.Conv2d(cin,cout,3,padding=1),nn.GroupNorm(groups,cout),nn.ReLU(),
            nn.Conv2d(cout,cout,3,padding=1),nn.GroupNorm(groups,cout),nn.ReLU())
    def forward(self,x): return self.seq(x)

class TinyUNet(nn.Module):
    def __init__(self,in_channels=3,out_channels=1,base=16):
        super().__init__()
        widths=[base,base*2,base*4,base*8]
        self.enc=nn.ModuleList([ConvBlock(in_channels,widths[0])]+[ConvBlock(widths[i-1],widths[i]) for i in range(1,4)])
        self.bottom=ConvBlock(widths[-1],base*16)
        self.dec=nn.ModuleList([ConvBlock(base*16+base*8,base*8),ConvBlock(base*8+base*4,base*4),
                               ConvBlock(base*4+base*2,base*2),ConvBlock(base*2+base,base)])
        self.head=nn.Conv2d(base,out_channels,1)
    def forward(self,x):
        if min(x.shape[-2:])<16: raise ValueError('U-Net 输入边长至少为 16。')
        skip=[]
        for block in self.enc:
            x=block(x); skip.append(x); x=F.max_pool2d(x,2)
        x=self.bottom(x)
        for block,s in zip(self.dec,reversed(skip)):
            x=F.interpolate(x,size=s.shape[-2:],mode='bilinear',align_corners=False)
            x=block(torch.cat([x,s],dim=1))
        return self.head(x)

class PatchDiscriminator(nn.Module):
    def __init__(self,base=16):
        super().__init__()
        layers=[nn.Conv2d(6,base,4,2,1),nn.LeakyReLU(.2)]
        for cin,cout,stride in [(base,base*2,2),(base*2,base*4,2),(base*4,base*8,1)]:
            layers += [nn.Conv2d(cin,cout,4,stride,1),nn.GroupNorm(4,cout),nn.LeakyReLU(.2)]
        layers += [nn.Conv2d(base*8,1,4,1,1)]
        self.seq=nn.Sequential(*layers)
    def forward(self,x,y): return self.seq(torch.cat([x,y],dim=1))


def weight_digest(model):
    h=hashlib.sha256()
    for k,v in model.state_dict().items():
        h.update(k.encode()); h.update(v.detach().cpu().numpy().tobytes())
    return h.hexdigest()


def save_checkpoint(path,model,meta,optimizer=None,extras=None):
    payload={'state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items()}, 'meta':safe_json(meta)}
    if optimizer is not None: payload['optimizer']=optimizer.state_dict()
    if extras: payload.update(extras)
    torch.save(payload,path)


def read_checkpoint(path):
    # 只加载可信来源；weights_only 避免普通 pickle 的任意对象反序列化。
    return torch.load(path,map_location='cpu',weights_only=True)


# ---- 两个基础项目 ----
class IndexedTransform(Dataset):
    def __init__(self,base,indices,kind='digit',size=128):
        self.base=base; self.indices=list(map(int,indices)); self.kind=kind; self.size=size
    def __len__(self): return len(self.indices)
    def __getitem__(self,i):
        im,target=self.base[self.indices[i]]
        if self.kind=='digit':
            a=np.asarray(im,dtype=np.float32)/255.
            return tensor_image(a),int(target),self.indices[i]
        im=im.convert('RGB').resize((self.size,self.size),Image.Resampling.BILINEAR)
        raw=np.asarray(target.resize((self.size,self.size),Image.Resampling.NEAREST))
        values=set(np.unique(raw).tolist())
        if not values.issubset({1,2,3}): raise ValueError(f'Oxford trimap 编码异常：{values}')
        # 1=宠物，2=背景，3=未确定边界；255 表示训练和评价都忽略。
        y=np.full(raw.shape,255,dtype=np.int64); y[raw==1]=1; y[raw==2]=0
        return tensor_image(np.asarray(im,dtype=np.float32)/255),torch.from_numpy(y),self.indices[i]


def fixed_indices(n,seed=42,val_fraction=.2,max_train=None,max_val=None):
    if n<3: raise ValueError('至少需要 3 条样本。')
    order=np.random.default_rng(seed).permutation(n)
    nv=max(1,min(n-1,int(round(n*val_fraction))))
    va=order[:nv]; tr=order[nv:]
    return tr[:max_train],va[:max_val]


def load_digit_data(root,download=True,seed=42,max_train=10000,max_val=2000,max_test=None):
    from torchvision.datasets import MNIST
    root=Path(root); root.mkdir(parents=True,exist_ok=True)
    a=MNIST(str(root),train=True,download=download); b=MNIST(str(root),train=False,download=download)
    tr,va=fixed_indices(len(a),seed,.1,max_train,max_val)
    te=np.arange(len(b))[:max_test]
    return IndexedTransform(a,tr),IndexedTransform(a,va),IndexedTransform(b,te)


def load_pet_data(root,download=True,seed=42,size=128,max_train=1000,max_val=200,max_test=400):
    from torchvision.datasets import OxfordIIITPet
    root=Path(root); root.mkdir(parents=True,exist_ok=True)
    a=OxfordIIITPet(str(root),split='trainval',target_types='segmentation',download=download)
    b=OxfordIIITPet(str(root),split='test',target_types='segmentation',download=download)
    tr,va=fixed_indices(len(a),seed,.2,max_train,max_val)
    # 固定子集；数据集没有由本课程核实的个体动物标识。
    te=np.random.default_rng(seed).permutation(len(b))[:max_test]
    return tuple(IndexedTransform(base,ids,'pet',size) for base,ids in [(a,tr),(a,va),(b,te)])


def masked_bce(logits,target):
    valid=target!=255
    if not bool(valid.any()): raise ValueError('该批次没有可评价像素。')
    z=logits.squeeze(1)
    return F.binary_cross_entropy_with_logits(z[valid],target[valid].float())


def binary_scores(pred,target):
    p=np.asarray(pred,dtype=bool); y=np.asarray(target)
    valid=y!=255
    if not valid.any(): return dict(dice=np.nan,iou=np.nan,tp=0,fp=0,fn=0,valid_pixels=0)
    truth=y==1
    tp=int((p & truth & valid).sum()); fp=int((p & ~truth & valid).sum()); fn=int((~p & truth & valid).sum())
    return dict(dice=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 1.,
                iou=tp/(tp+fp+fn) if tp+fp+fn else 1.,tp=tp,fp=fp,fn=fn,valid_pixels=int(valid.sum()))


def train_supervised(model,train_ds,val_ds,folder,task,epochs=3,batch_size=64,lr=.001,seed=42):
    folder=Path(folder); folder.mkdir(parents=True,exist_ok=True)
    dev=device_for_training(); model.to(dev); opt=torch.optim.Adam(model.parameters(),lr=lr)
    train_loader=loader(train_ds,batch_size,True,seed); val_loader=loader(val_ds,batch_size,False,seed)
    history=[]; best=float('inf')
    for epoch in range(1,epochs+1):
        record={'epoch':epoch}
        for phase,dl in [('train',train_loader),('val',val_loader)]:
            model.train(phase=='train'); total=0.; count=0
            with torch.set_grad_enabled(phase=='train'):
                for x,y,_ in dl:
                    x,y=x.to(dev),y.to(dev)
                    z=model(x); loss=F.cross_entropy(z,y) if task=='digit' else masked_bce(z,y)
                    finite_loss(loss)
                    if phase=='train': opt.zero_grad(); loss.backward(); opt.step()
                    total+=float(loss.detach())*len(x); count+=len(x)
            record[phase+'_loss']=total/count
        history.append(record); print(record)
        if record['val_loss']<best:
            best=record['val_loss']; save_checkpoint(folder/'best.pt',model,dict(task=task,epoch=epoch,best_val=best,seed=seed),opt)
        pd.DataFrame(history).to_csv(folder/'history.csv',index=False)
    model.load_state_dict(read_checkpoint(folder/'best.pt')['state_dict']); model.eval()
    save_curve(history,['train_loss','val_loss'],folder/'learning_curve.png','Loss')
    return model,history


@torch.inference_mode()
def evaluate_digits(model,dataset,folder,batch_size=128):
    dev=next(model.parameters()).device; model.eval(); rows=[]
    for x,y,ids in loader(dataset,batch_size):
        prob=model(x.to(dev)).softmax(1).cpu(); pred=prob.argmax(1)
        rows.extend(dict(sample_id=int(i),truth=int(t),prediction=int(p),confidence=float(c))
                    for i,t,p,c in zip(ids,y,pred,prob.max(1).values))
    df=pd.DataFrame(rows); p=Path(folder); p.mkdir(parents=True,exist_ok=True)
    df.to_csv(p/'test_predictions.csv',index=False)
    cm=confusion_matrix(df.truth,df.prediction,labels=list(range(10)))
    pd.DataFrame(cm,index=range(10),columns=range(10)).to_csv(p/'confusion_matrix.csv')
    scores=dict(n=len(df),accuracy=accuracy_score(df.truth,df.prediction),macro_f1=f1_score(df.truth,df.prediction,average='macro',labels=list(range(10)),zero_division=0))
    write_json(p/'metrics.json',scores)
    fig,ax=plt.subplots(figsize=(5,5)); ax.imshow(cm); ax.set(xlabel='Predicted',ylabel='Reference',xticks=range(10),yticks=range(10))
    fig.tight_layout();fig.savefig(p/'confusion_matrix.png',dpi=180);plt.close(fig)
    return df,scores


@torch.inference_mode()
def evaluate_pets(model,dataset,folder,batch_size=16,threshold=.5):
    model.eval(); dev=next(model.parameters()).device; rows=[]
    for x,y,ids in loader(dataset,batch_size):
        p=(model(x.to(dev)).sigmoid().squeeze(1).cpu().numpy()>=threshold)
        for k,truth,idx in zip(p,y.numpy(),ids): rows.append(dict(sample_id=int(idx),**binary_scores(k,truth)))
    df=pd.DataFrame(rows); folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    df.to_csv(folder/'test_predictions.csv',index=False)
    tp,fp,fn=df[['tp','fp','fn']].sum()
    scores=dict(n=len(df),mean_image_dice=float(df.dice.mean()),mean_image_iou=float(df.iou.mean()),
       global_dice=float(2*tp/(2*tp+fp+fn)) if 2*tp+fp+fn else 1.,threshold=threshold,ignored_raw_label=3)
    write_json(folder/'metrics.json',scores);return df,scores

# ---- 真实 BCI 数据的配对、审计和划分 ----
def image_map(folder):
    p=Path(folder)
    paths=sorted(x for x in p.rglob('*') if x.is_file() and x.suffix.lower() in IMAGE_SUFFIXES)
    out={}
    for f in paths:
        key=f.relative_to(p).with_suffix('').as_posix()
        if key in out: raise ValueError(f'同一配对键对应多个文件：{key}')
        out[key]=f
    return out


def discover_bci(root,layout='auto_separate',he_side=None):
    """只按明确目录语义配对；组合图必须显式指定左右染色方向。"""
    root=Path(root).resolve()
    if not root.exists(): raise FileNotFoundError(f'BCI 数据目录不存在：{root}')
    schemes=[('HE/train','IHC/train','HE/test','IHC/test'),
             ('he/train','ihc/train','he/test','ihc/test'),
             ('train/HE','train/IHC','test/HE','test/IHC'),
             ('train/he','train/ihc','test/he','test/ihc')]
    if layout=='a_to_b': schemes=[('trainA','trainB','testA','testB'),('A/train','B/train','A/test','B/test')]
    elif layout=='b_to_a': schemes=[('trainB','trainA','testB','testA'),('B/train','A/train','B/test','A/test')]
    elif layout not in {'auto_separate','combined'}: raise ValueError('layout 应为 auto_separate/a_to_b/b_to_a/combined。')
    rows=[]
    if layout=='combined':
        if he_side not in {'left','right'}: raise ValueError('组合图必须设置 he_side="left" 或 "right"，并视觉检查方向。')
        for split in ['train','test']:
            ims=image_map(root/split)
            if not ims: raise ValueError(f'{split} 中没有组合图。')
            for k,p in ims.items(): rows.append(dict(pair_id=f'{split}/{k}',source_split=split,
                he_path=p.relative_to(root).as_posix(),ihc_path=p.relative_to(root).as_posix(),layout='combined',he_side=he_side))
    else:
        matches=[s for s in schemes if all((root/d).is_dir() for d in s)]
        if len(matches)!=1: raise ValueError(f'找到 {len(matches)} 种可识别布局。请让 BCI_ROOT 指向包含 HE/IHC 或 trainA/trainB 的直接父目录，并明确 LAYOUT。')
        for split,(a,b) in zip(['train','test'],[(matches[0][0],matches[0][1]),(matches[0][2],matches[0][3])]):
            he,ihc=image_map(root/a),image_map(root/b)
            if not he or not ihc: raise ValueError(f'{split} 文件夹为空。')
            if set(he)!=set(ihc):
                raise ValueError(f'{split} 配对不完整；仅H&E：{sorted(set(he)-set(ihc))[:5]}；仅IHC：{sorted(set(ihc)-set(he))[:5]}。请依据来源修复后重跑。')
            for k in sorted(he): rows.append(dict(pair_id=f'{split}/{k}',source_split=split,
                he_path=he[k].relative_to(root).as_posix(),ihc_path=ihc[k].relative_to(root).as_posix(),layout='separate',he_side=''))
    df=pd.DataFrame(rows)
    import re
    def score(p):
        m=re.search(r'(?:_|-)([0-3])\+?$',Path(p).stem)
        return m.group(1)+('+' if m and m.group(1)!='0' else '') if m else 'unknown'
    df['her2_label']=df.he_path.map(score)
    return df


def resolve_bci_root(explicit='',layout='auto_separate',he_side=None):
    if explicit:
        p=Path(explicit); discover_bci(p,layout,he_side); return p.resolve()
    candidates=[]
    for parent in [Path('/kaggle/input'),Path('./data')]:
        if not parent.exists(): continue
        candidates.append(parent)
        candidates.extend(p for p in parent.glob('*') if p.is_dir())
        candidates.extend(p for p in parent.glob('*/*') if p.is_dir())
        candidates.extend(p for p in parent.glob('*/*/*') if p.is_dir())
    found=[]
    for p in candidates:
        try: discover_bci(p,layout,he_side); found.append(p.resolve())
        except (ValueError,FileNotFoundError): pass
    found=sorted(set(found))
    if len(found)!=1:
        raise FileNotFoundError(f'找到 {len(found)} 个 BCI 数据根目录：{found}。请按数据章节取得原始数据、解压并添加为 Kaggle Input，再填写 BCI_ROOT。代码不会用模拟图像替代缺失的病理数据。')
    return found[0]


def find_artifact(filename,explicit='',preferred=None):
    if explicit:
        p=Path(explicit)
        if not p.is_file(): raise FileNotFoundError(f'找不到 {p}')
        return p.resolve()
    if preferred is not None and Path(preferred).is_file(): return Path(preferred).resolve()
    found=[]
    for root in [Path('/kaggle/input'),work_root()]:
        if root.exists(): found.extend(root.rglob(filename))
    found=sorted(set(p.resolve() for p in found))
    if len(found)!=1: raise FileNotFoundError(f'找到 {len(found)} 个 {filename}：{found[:8]}。请把上一步输出添加为 Input，并填写文件路径。')
    return found[0]


def read_pair(row,root):
    root=Path(root).resolve()
    def checked_path(rel):
        p=(root/str(rel)).resolve()
        if not p.is_relative_to(root): raise ValueError('清单中的路径越出了数据根目录。')
        return p
    he=read_rgb(checked_path(row['he_path']))
    if row['layout']=='combined':
        w,h=he.size
        if w%2: raise ValueError(f"组合图宽度不是偶数：{row['pair_id']}")
        left=he.crop((0,0,w//2,h)); right=he.crop((w//2,0,w,h))
        he,ihc=(left,right) if row['he_side']=='left' else (right,left)
    else: ihc=read_rgb(checked_path(row['ihc_path']))
    if he.size!=ihc.size: raise ValueError(f"配对尺寸不一致：{row['pair_id']} {he.size} / {ihc.size}")
    return he,ihc


def audit_bci(df,root):
    records=[]
    for row in df.to_dict('records'):
        x,y=read_pair(row,root)
        a,b=np.asarray(x),np.asarray(y)
        h1=hashlib.sha256(a.tobytes()+str(a.shape).encode()).hexdigest()
        h2=hashlib.sha256(b.tobytes()+str(b.shape).encode()).hexdigest()
        records.append(dict(pair_id=row['pair_id'],width=x.width,height=x.height,he_sha256=h1,ihc_sha256=h2,
             reference_nonwhite_fraction=float((b.min(2)<230).mean())))
    return df.merge(pd.DataFrame(records),on='pair_id',validate='one_to_one')


def assign_bci_splits(df,metadata_csv=None,seed=42,val_fraction=.2,allow_patch_validation=True):
    df=df.copy()
    if metadata_csv:
        meta=pd.read_csv(metadata_csv,dtype=str).fillna('')
        if 'pair_id' not in meta or meta.pair_id.duplicated().any(): raise ValueError('metadata.csv 必须有唯一 pair_id。')
        level='patient_id' if 'patient_id' in meta and meta.patient_id.ne('').all() else 'slide_id' if 'slide_id' in meta and meta.slide_id.ne('').all() else None
        if not level: raise ValueError('需要完整、经来源核实的 patient_id 或 slide_id；空值不以文件名前缀代替。')
        df=df.merge(meta[['pair_id',level]],on='pair_id',how='left',validate='one_to_one')
        if df[level].isna().any() or df[level].eq('').any(): raise ValueError('元数据没有覆盖全部配对。')
        df['group_id']=df[level];df['group_level']=level
    else:
        if not allow_patch_validation: raise ValueError('缺少切片/患者标识，已禁止图块级验证。')
        df['group_id']='';df['group_level']='unknown'
    df['split']=df.source_split
    tr=df.source_split.eq('train')
    if df.group_level.iloc[0]!='unknown':
        tr_groups=set(df.loc[tr,'group_id']); test_groups=set(df.loc[~tr,'group_id'])
        overlap=tr_groups & test_groups
        if overlap: raise ValueError(f'来源官方 train/test 存在组重叠：{sorted(overlap)[:8]}。需要先另行制定并记录组级划分。')
        groups=np.array(sorted(tr_groups))
        if len(groups)<2: raise ValueError('训练来源至少需要两个独立组才能留出验证集。')
        order=np.random.default_rng(seed).permutation(groups)
        n=max(1,min(len(order)-1,round(len(order)*val_fraction)))
        va_groups=set(order[:n]);df.loc[tr & df.group_id.isin(va_groups),'split']='val'
    else:
        # 只在官方训练来源中留出验证；此模式的独立性范围明确标记为未核实。
        ids=df.loc[tr,'pair_id'].tolist()
        order=sorted(ids,key=lambda s:hashlib.sha256(f'{seed}:{s}'.encode()).hexdigest())
        n=max(1,min(len(order)-1,round(len(order)*val_fraction)))
        df.loc[df.pair_id.isin(order[:n]),'split']='val'
    if set(df.split)!={'train','val','test'}: raise ValueError('划分必须含非空 train/val/test。')
    # 全量源数据先查重，再选择课程子集；不靠选子集绕开泄漏。
    for col in ['he_sha256','ihc_sha256']:
        if col in df:
            bad=df.groupby(col).split.nunique()
            if (bad>1).any():
                raise ValueError(f'{col} 发现跨集合完全相同图像。修复数据来源/划分后再继续。')
    return df


def subset_manifest(df,limits=None,seed=42):
    if not limits: return df.sort_values('pair_id').reset_index(drop=True)
    chunks=[]
    for split in ['train','val','test']:
        part=df[df.split==split].copy()
        part['_order']=part.pair_id.map(lambda p:hashlib.sha256(f'{seed}:subset:{p}'.encode()).hexdigest())
        chunks.append(part.sort_values('_order').head(limits.get(split,len(part))).drop(columns='_order'))
    return pd.concat(chunks).sort_values('pair_id').reset_index(drop=True)


def prepare_bci(root,folder,layout='auto_separate',he_side=None,metadata_csv=None,seed=42,limits=None):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    df=audit_bci(discover_bci(root,layout,he_side),root)
    df=assign_bci_splits(df,metadata_csv,seed)
    df.to_csv(folder/'bci_full_audit.csv',index=False)
    selected=subset_manifest(df,limits,seed)
    selected.to_csv(folder/'bci_manifest.csv',index=False)
    info=dict(source='BCI',marker='HER2',root_hint=str(Path(root).resolve()),layout=layout,he_side=he_side,
        n_full=len(df),n_selected=len(selected),split_counts=selected.split.value_counts().to_dict(),seed=seed,
        group_level=selected.group_level.iloc[0],patient_independence_verified=bool(selected.group_level.iloc[0]=='patient_id'),
        selection='deterministic SHA256 ordering within each split',limits=limits,
        manifest_sha256=sha_file(folder/'bci_manifest.csv'),audit_sha256=sha_file(folder/'bci_full_audit.csv'),
        reference='https://github.com/bupt-ai-cz/BCI',course_version=VERSION)
    write_json(folder/'data_card.json',info)
    return selected,info


def load_manifest(path):
    df=pd.read_csv(path,keep_default_na=False,dtype={k:str for k in ['pair_id','he_path','ihc_path','layout','he_side','split','group_id','group_level','her2_label','he_sha256','ihc_sha256']})
    levels=set(df.group_level) if 'group_level' in df else set()
    if len(levels)!=1 or not levels.issubset({'unknown','patient_id','slide_id'}): raise ValueError('group_level必须是统一的unknown、patient_id或slide_id。')
    needed={'pair_id','he_path','ihc_path','layout','he_side','split','group_id','group_level'}
    if not needed.issubset(df): raise ValueError(f'清单缺少字段：{needed-set(df)}')
    if df.pair_id.duplicated().any(): raise ValueError('pair_id 不唯一。')
    if set(df.split)!={'train','val','test'}: raise ValueError('清单需包含 train/val/test。')
    if set(df.group_level)!={'unknown'}:
        if df.group_id.eq('').any(): raise ValueError('组标识不完整。')
        if (df.groupby('group_id').split.nunique()>1).any(): raise ValueError('组跨集合重叠。')
    return df

class BCIPairs(Dataset):
    def __init__(self,manifest,root,split,size=256,augment=False,seed=42):
        self.rows=manifest[manifest.split==split].to_dict('records'); self.root=Path(root)
        self.size=int(size);self.augment=augment;self.seed=seed;self.epoch=0
        if not self.rows: raise ValueError(f'{split} 数据为空。')
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows[i];x,y=read_pair(row,self.root)
        x=x.resize((self.size,self.size),Image.Resampling.BILINEAR)
        y=y.resize((self.size,self.size),Image.Resampling.BILINEAR)
        if self.augment:
            n=int(hashlib.sha256(f"{self.seed}:{self.epoch}:{row['pair_id']}".encode()).hexdigest()[:16],16)
            rng=random.Random(n)
            if rng.random()<.5: x,y=ImageOps.mirror(x),ImageOps.mirror(y)
            if rng.random()<.5: x,y=ImageOps.flip(x),ImageOps.flip(y)
        return tensor_image(np.asarray(x,dtype=np.float32)/255.),tensor_image(np.asarray(y,dtype=np.float32)/255.),row['pair_id']


def verify_data_files(manifest,root,check_hashes=False):
    """检查清单和挂载的实际数据；跨 Notebook 重新挂载后可开启完整哈希核验。"""
    for row in manifest.to_dict('records'):
        x,y=read_pair(row,root)
        if check_hashes:
            for im,key in [(x,'he_sha256'),(y,'ihc_sha256')]:
                if key not in row: raise ValueError('清单缺少原图哈希。')
                a=np.asarray(im);h=hashlib.sha256(a.tobytes()+str(a.shape).encode()).hexdigest()
                if h!=row[key]: raise ValueError(f"图像已变化：{row['pair_id']} {key}")
    return True


@dataclass
class StainConfig:
    size:int=256
    base:int=16
    epochs:int=10
    batch_size:int=8
    lr:float=0.0002
    seed:int=42
    lambda_l1:float=100.0
    lambda_gan:float=1.0
    augment:bool=True


def fit_color_baseline(train_ds,folder,max_pairs=64,pixels_per_pair=1024,seed=42):
    """只用训练集拟合 RGB 到 RGB 的线性映射；用于检验全局颜色变换的能力。"""
    rng=np.random.default_rng(seed); xs=[];ys=[]
    for i in range(min(max_pairs,len(train_ds))):
        x,y,_=train_ds[i];x=numpy_image(x).reshape(-1,3);y=numpy_image(y).reshape(-1,3)
        ix=rng.choice(len(x),size=min(pixels_per_pair,len(x)),replace=False)
        xs.append(np.column_stack([x[ix],np.ones(len(ix))]));ys.append(y[ix])
    X=np.concatenate(xs).astype(np.float64);Y=np.concatenate(ys).astype(np.float64)
    penalty=np.eye(4)*.001;penalty[-1,-1]=0
    weights=np.linalg.solve(X.T@X+penalty,X.T@Y)
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True);np.save(folder/'color_weights.npy',weights)
    write_json(folder/'color_fit.json',dict(training_pairs=min(max_pairs,len(train_ds)),pixels_per_pair=pixels_per_pair,seed=seed))
    return weights


def apply_color(x,weights):
    shape=x.shape
    return np.clip(np.column_stack([x.reshape(-1,3),np.ones(shape[0]*shape[1])])@weights,0,1).reshape(shape).astype(np.float32)


@torch.inference_mode()
def val_stain_mae(model,dataset,batch_size=8):
    model.eval();dev=next(model.parameters()).device;total=0.;n=0
    for x,y,_ in loader(dataset,batch_size):
        pred=model(x.to(dev)).sigmoid(); total+=float(F.l1_loss(pred,y.to(dev)))*len(x);n+=len(x)
    return total/n


def train_stain(manifest,root,folder,cfg,mode='l1',manifest_hash=''):
    if mode not in {'l1','pix2pix'}:raise ValueError('mode 只能是 l1 或 pix2pix。')
    if cfg.size<32: raise ValueError('虚拟染色输入边长至少为32。')
    if cfg.epochs<1 or cfg.batch_size<1 or cfg.lr<=0 or cfg.base<4 or cfg.base%4: raise ValueError('epochs/batch_size/lr必须为正；base使用不小于4的4的倍数。')
    if cfg.lambda_l1<=0 or cfg.lambda_gan<0: raise ValueError('L1权重必须为正，GAN权重不能为负。')
    seed_everything(cfg.seed);dev=device_for_training()
    out=Path(folder)/mode;out.mkdir(parents=True,exist_ok=True)
    if (out/'best.pt').exists(): raise FileExistsError(f'{out} 已有训练结果，请建立新的实验目录。')
    model=TinyUNet(3,3,cfg.base).to(dev);initial=weight_digest(model)
    discriminator=PatchDiscriminator(cfg.base).to(dev) if mode=='pix2pix' else None
    opt_g=torch.optim.Adam(model.parameters(),lr=cfg.lr,betas=(.5,.999))
    opt_d=torch.optim.Adam(discriminator.parameters(),lr=cfg.lr,betas=(.5,.999)) if discriminator else None
    tr=BCIPairs(manifest,root,'train',cfg.size,cfg.augment,cfg.seed)
    va=BCIPairs(manifest,root,'val',cfg.size,False,cfg.seed)
    dl=loader(tr,cfg.batch_size,True,cfg.seed)
    history=[];best=float('inf');start=time.time()
    meta=dict(task='he_to_her2_ihc',marker='HER2',model='TinyUNet',mode=mode,config=asdict(cfg),
        manifest_sha256=manifest_hash,initial_weights_sha256=initial,
        checkpoint_selection='minimum validation RGB MAE',preprocess='whole paired patch resized bilinearly to size; RGB/255',
        group_level=manifest.group_level.iloc[0],source='BCI',course_version=VERSION)
    for epoch in range(1,cfg.epochs+1):
        model.train();tr.epoch=epoch
        if discriminator: discriminator.train()
        totals=dict(train_l1=0.,train_g=0.,train_d=0.);n=0
        for x,y,_ in dl:
            x,y=x.to(dev),y.to(dev);pred=model(x).sigmoid()
            d_loss=torch.tensor(0.,device=dev)
            if discriminator:
                for p in discriminator.parameters():p.requires_grad_(True)
                opt_d.zero_grad()
                dr=discriminator(x,y);df=discriminator(x,pred.detach())
                # Least-squares GAN：两种模式保留同样的像素项，附加项为对抗损失。
                d_loss=.5*((dr-1).square().mean()+df.square().mean())
                finite_loss(d_loss);d_loss.backward();opt_d.step()
                for p in discriminator.parameters():p.requires_grad_(False)
            opt_g.zero_grad()
            l1=F.l1_loss(pred,y);g_loss=cfg.lambda_l1*l1
            if discriminator:g_loss=g_loss+cfg.lambda_gan*(discriminator(x,pred)-1).square().mean()
            finite_loss(g_loss);g_loss.backward();opt_g.step()
            totals['train_l1']+=float(l1.detach())*len(x);totals['train_g']+=float(g_loss.detach())*len(x)
            totals['train_d']+=float(d_loss.detach())*len(x);n+=len(x)
        val=val_stain_mae(model,va,cfg.batch_size)
        record=dict(epoch=epoch,**{k:v/n for k,v in totals.items()},val_mae=val)
        history.append(record);pd.DataFrame(history).to_csv(out/'history.csv',index=False)
        current={**meta,'epoch':epoch,'val_mae':val,'elapsed_seconds':time.time()-start}
        extras={}
        if discriminator:extras={'discriminator':discriminator.state_dict(),'optimizer_d':opt_d.state_dict()}
        save_checkpoint(out/'last.pt',model,current,opt_g,extras)
        if val<best:
            best=val;save_checkpoint(out/'best.pt',model,current)
        print(mode,record)
    save_curve(history,['train_l1','val_mae'],out/'mae_curve.png','RGB MAE')
    write_json(out/'training_record.json',{**meta,'best_val_mae':best,'epochs_completed':len(history),'best_checkpoint_sha256':sha_file(out/'best.pt'),'environment':environment_record(),'elapsed_seconds':time.time()-start})
    model.load_state_dict(read_checkpoint(out/'best.pt')['state_dict']);model.eval()
    return model,history


def load_stain_model(checkpoint,expected_manifest_hash=None):
    data=read_checkpoint(checkpoint);meta=data['meta']
    if meta.get('task')!='he_to_her2_ihc': raise ValueError('权重任务不匹配，需要本课程的 H&E→HER2-IHC 权重。')
    if expected_manifest_hash and meta.get('manifest_sha256')!=expected_manifest_hash:raise ValueError('权重与数据划分清单的哈希不一致。')
    model=TinyUNet(3,3,meta['config']['base']);model.load_state_dict(data['state_dict']);model.to(device_for_training());model.eval()
    return model,meta

# ---- 统一评价：像素误差、结构相似性、DAB 颜色分量 ----
def tissue_mask(reference):
    return np.min(reference,axis=-1)<.90


def dab_component(rgb):
    # 固定 HED 解混矩阵得到的相对颜色分量，数值不等于蛋白浓度。
    return np.maximum(rgb2hed(np.clip(rgb,1e-6,1.0))[...,2],0.)


def calibrate_dab(val_ds,output,max_pairs=64,seed=42):
    rng=np.random.default_rng(seed);values=[];ids=[]
    for i in range(min(max_pairs,len(val_ds))):
        _,y,pid=val_ds[i];a=numpy_image(y);m=tissue_mask(a);v=dab_component(a)[m]
        if len(v):values.append(v[rng.choice(len(v),min(2048,len(v)),replace=False)])
        ids.append(pid)
    if not values: raise ValueError('验证集未找到非白背景区域，无法标定 DAB 颜色阈值。')
    z=np.concatenate(values)
    threshold=float(threshold_otsu(z)) if np.ptp(z)>1e-12 else float(z[0]+1e-6)
    obj=dict(threshold=threshold,method='Otsu on sampled validation-reference DAB component',
             reference_ids=ids,seed=seed,tissue_rule='minimum RGB channel < 0.90',
             interpretation='relative DAB color proxy; not clinical positivity or HER2 score')
    write_json(output,obj);return obj


def reconstruction_metrics(pred,reference,dab_threshold):
    p=np.asarray(pred,dtype=np.float64);y=np.asarray(reference,dtype=np.float64)
    if p.shape!=y.shape or p.ndim!=3 or p.shape[-1]!=3:raise ValueError('评价需要同形状 H×W×3 RGB 数组。')
    if not np.isfinite(p).all() or not np.isfinite(y).all():raise ValueError('图像出现非有限值。')
    if min(p.min(),y.min())<0 or max(p.max(),y.max())>1:raise ValueError('评价图像必须统一在 [0,1] 范围。')
    err=np.abs(p-y);m=tissue_mask(y);mse=float(((p-y)**2).mean())
    psnr=float('inf') if mse==0 else float(10*np.log10(1/mse))
    win=min(7,min(y.shape[:2]));win=win if win%2 else win-1
    if win<3:raise ValueError('SSIM 图像边长至少为3。')
    ssim=float(structural_similarity(y,p,data_range=1.,channel_axis=2,win_size=win))
    dy,dp=dab_component(y),dab_component(p)
    result=dict(rgb_mae=float(err.mean()),psnr=psnr,ssim=ssim,tissue_pixels=int(m.sum()),
        tissue_mae=float(err[m].mean()) if m.any() else np.nan,
        dab_mae=float(np.abs(dp[m]-dy[m]).mean()) if m.any() else np.nan,
        dab_mean_ref=float(dy[m].mean()) if m.any() else np.nan,
        dab_mean_pred=float(dp[m].mean()) if m.any() else np.nan,
        dab_area_ref=float((dy[m]>dab_threshold).mean()) if m.any() else np.nan,
        dab_area_pred=float((dp[m]>dab_threshold).mean()) if m.any() else np.nan)
    result['dab_area_abs_error']=abs(result['dab_area_pred']-result['dab_area_ref'])
    return result


def perturb_input(x,condition):
    if condition=='clean':return x.copy()
    if condition=='blur_sigma1':return np.clip(gaussian_filter(x,sigma=(1.,1.,0.)),0,1).astype(np.float32)
    raise ValueError(f'未登记的条件：{condition}')


@torch.inference_mode()
def predict_rgb(model,x):
    t=tensor_image(x).unsqueeze(0).to(next(model.parameters()).device)
    return numpy_image(model(t).sigmoid()[0])


def paired_summary(df,metric='rgb_mae',a='l1',b='pix2pix',condition='clean',bootstrap=2000,seed=42):
    rows=df[df.condition.eq(condition)]
    left=rows[rows.method.eq(a)].set_index('pair_id');right=rows[rows.method.eq(b)].set_index('pair_id')
    if set(left.index)!=set(right.index): raise ValueError('两个方法的评价样本不一致。')
    out=pd.DataFrame({'delta':right[metric]-left[metric],'group_id':left.group_id,'group_level':left.group_level}).dropna(subset=['delta'])
    result=dict(metric=metric,contrast=f'{b} - {a}',condition=condition,n_pairs=len(out),
                mean_delta=float(out.delta.mean()),ci95=None,inference_unit='patch_descriptive')
    known=len(out)>0 and set(out.group_level)!={'unknown'} and out.group_id.ne('').all()
    if known:
        units=out.groupby('group_id').delta.mean().to_numpy()
        result.update(inference_unit=out.group_level.iloc[0],n_groups=len(units),group_mean_delta=float(units.mean()))
        if len(units)>=5:
            rng=np.random.default_rng(seed)
            means=np.array([rng.choice(units,len(units),replace=True).mean() for _ in range(bootstrap)])
            result['ci95']=list(map(float,np.quantile(means,[.025,.975])))
            result['ci_note']='equal-weight group means; descriptive bootstrap interval'
    return result


def evaluate_stain(manifest,root,run_dir,output,conditions=('clean','blur_sigma1'),max_examples=3):
    run_dir=Path(run_dir);output=Path(output);output.mkdir(parents=True,exist_ok=True)
    config=json.loads((run_dir/'experiment.json').read_text(encoding='utf-8'))
    expected=config['manifest_sha256']
    local_copy=run_dir/'bci_manifest.csv'
    if sha_file(local_copy)!=expected:raise ValueError('实验目录中的清单已变化。')
    # 外部清单和权重使用同一对样本，逐字段核对避免挂错版本。
    frozen=load_manifest(local_copy)
    if manifest.to_csv(index=False)!=frozen.to_csv(index=False):raise ValueError('本次评价清单与冻结训练清单不一致。')
    size=config['config']['size'];va=BCIPairs(manifest,root,'val',size)
    te=BCIPairs(manifest,root,'test',size)
    calibration=calibrate_dab(va,output/'dab_calibration.json')
    models={};metas={}
    for mode in ['l1','pix2pix']:
        models[mode],metas[mode]=load_stain_model(run_dir/mode/'best.pt',expected)
        if metas[mode]['config']!=config['config']:raise ValueError('检查点配置与experiment.json不一致。')
    if metas['l1']['initial_weights_sha256']!=metas['pix2pix']['initial_weights_sha256']:
        raise ValueError('对照实验生成器初始权重不同。请检查随机种子和模型配置。')
    weights=np.load(run_dir/'color_weights.npy',allow_pickle=False)
    source=manifest.set_index('pair_id');records=[]
    # 样例编号在看测试分数前固定；另有按误差排序的失败案例列表。
    show_ids=set(sorted(r['pair_id'] for r in te.rows)[:max_examples])
    for i in range(len(te)):
        xt,yt,pid=te[i];x=numpy_image(xt);y=numpy_image(yt)
        for condition in conditions:
            xx=perturb_input(x,condition)
            preds={'identity':xx,'color_linear':apply_color(xx,weights)}
            preds.update({name:predict_rgb(model,xx) for name,model in models.items()})
            for name,pred in preds.items():
                stats=reconstruction_metrics(pred,y,calibration['threshold'])
                records.append(dict(pair_id=pid,method=name,condition=condition,group_id=source.loc[pid,'group_id'],
                    group_level=source.loc[pid,'group_level'],her2_label=source.loc[pid,'her2_label'],**stats))
            if pid in show_ids and condition=='clean':
                safe=hashlib.sha256(pid.encode()).hexdigest()[:12]
                dest=output/'examples'/safe;dest.mkdir(parents=True,exist_ok=True)
                for name,arr in {'he':x,'reference_ihc':y,**preds}.items():
                    Image.fromarray(np.round(np.clip(arr,0,1)*255).astype('uint8')).save(dest/f'{name}.png')
                write_json(dest/'provenance.json',dict(pair_id=pid,condition=condition,prediction_label='generated HER2-IHC; research only'))
    df=pd.DataFrame(records);df.to_csv(output/'per_pair_metrics.csv',index=False)
    metric_cols=['rgb_mae','tissue_mae','psnr','ssim','dab_mae','dab_area_abs_error']
    summary=[]
    for (method,condition),part in df.groupby(['method','condition'],sort=False):
        record=dict(method=method,condition=condition,n_pairs=len(part))
        for col in metric_cols:
            vals=part[col].replace([np.inf,-np.inf],np.nan)
            record[col+'_mean']=float(vals.mean());record[col+'_std']=float(vals.std())
            record[col+'_finite_n']=int(vals.notna().sum())
            record[col+'_infinite_n']=int(np.isinf(part[col]).sum())
        summary.append(record)
    sf=pd.DataFrame(summary);sf.to_csv(output/'summary.csv',index=False)
    contrast=paired_summary(df);write_json(output/'paired_comparison.json',contrast)
    clean=df[df.condition.eq('clean')]
    failed=clean[clean.method.eq('l1')].sort_values(['rgb_mae','pair_id'],ascending=[False,True]).head(10)
    failed.to_csv(output/'failure_candidates.csv',index=False)
    clean.groupby(['method','her2_label'])[metric_cols].agg(['mean','count']).to_csv(output/'source_label_descriptive.csv')
    if manifest.group_level.iloc[0]!='unknown':
        df.groupby(['method','condition','group_id'])[metric_cols].mean().to_csv(output/'per_group_metrics.csv')
    for metric in ['rgb_mae','ssim','dab_mae']:
        rows=sf[sf.condition.eq('clean')];fig,ax=plt.subplots(figsize=(7,4))
        ax.bar(rows.method,rows[metric+'_mean']);ax.set(ylabel=metric,xlabel='Method')
        fig.tight_layout();fig.savefig(output/f'{metric}_comparison.png',dpi=180);plt.close(fig)
    if len(conditions)>1:
        pivot=df[df.method.eq('l1')].pivot(index='pair_id',columns='condition',values='rgb_mae')
        if {'clean','blur_sigma1'}.issubset(pivot):
            fig,ax=plt.subplots(figsize=(5,5));ax.scatter(pivot.clean,pivot.blur_sigma1,s=12)
            lo=float(pivot.min().min());hi=float(pivot.max().max());ax.plot([lo,hi],[lo,hi])
            ax.set(xlabel='Clean RGB MAE',ylabel='Blurred-input RGB MAE');fig.tight_layout()
            fig.savefig(output/'blur_paired_scatter.png',dpi=180);plt.close(fig)
    write_json(output/'evaluation_record.json',dict(manifest_sha256=expected,config=config['config'],conditions=list(conditions),
        checkpoint_sha256={m:sha_file(run_dir/m/'best.pt') for m in models},
        calibration_sha256=sha_file(output/'dab_calibration.json'),color_weights_sha256=sha_file(run_dir/'color_weights.npy'),
        test_pairs=len(te),group_level=manifest.group_level.iloc[0],environment=environment_record(),
        missing_metric_handling='NaN written as empty CSV field; infinite PSNR counted separately; finite denominators recorded'))
    return df,sf,contrast


def materialize_result_text(run_dir,eval_dir,output):
    run_dir=Path(run_dir);ev=Path(eval_dir);output=Path(output);output.mkdir(parents=True,exist_ok=True)
    summary=pd.read_csv(ev/'summary.csv');record=json.loads((ev/'evaluation_record.json').read_text())
    exp=json.loads((run_dir/'experiment.json').read_text());mf=load_manifest(run_dir/'bci_manifest.csv')
    table=summary[summary.condition.eq('clean')][['method','n_pairs','rgb_mae_mean','tissue_mae_mean','ssim_mean','dab_mae_mean']]
    # 手动转 Markdown，不依赖 tabulate。
    headers=list(table.columns);lines=['| '+' | '.join(headers)+' |','| '+' | '.join(['---']*len(headers))+' |']
    for row in table.itertuples(index=False,name=None):
        lines.append('| '+' | '.join(f'{v:.6f}' if isinstance(v,float) else str(v) for v in row)+' |')
    a=table.set_index('method').loc['l1','rgb_mae_mean'];b=table.set_index('method').loc['pix2pix','rgb_mae_mean']
    change='较低' if b<a else '较高' if b>a else '相同'
    text=f'''# 本次实验的结果材料

## 数据与配置
本次使用 BCI 的 H&E—HER2-IHC 配对图像，训练、验证、测试图块数量分别为 {sum(mf.split=='train')}、{sum(mf.split=='val')}、{sum(mf.split=='test')}。图块整体缩放为 {record['config']['size']} × {record['config']['size']} 像素，训练 {record['config']['epochs']} 轮，随机种子为 {record['config']['seed']}。划分组标识层级为 `{record['group_level']}`。U-Net 使用验证集 RGB MAE 选择权重。数据清单哈希为 `{record['manifest_sha256']}`。

## 定量结果
{chr(10).join(lines)}

在该测试集上，pix2pix 教学实现的平均 RGB MAE 为 {b:.6f}，L1 基线为 {a:.6f}，前者{change}。差值为 {b-a:+.6f}。该比较对应当前数据子集、训练预算与实现。DAB 指标描述固定颜色解混后的相对分量，需结合组织位置和配对误差解释。

## 图表来源
`rgb_mae_comparison.png`、`ssim_comparison.png` 与 `dab_mae_comparison.png` 来自同一份 `per_pair_metrics.csv`。`examples/` 为预先固定样例；`failure_candidates.csv` 列出按 L1 基线误差排序的复查图块。

## 需要结合图像完成的观察
请在自己的报告中写出具体图块编号、观察位置以及两种模型之间的差异。检查棕色区域、核结构、组织边界和空白区域。将可能的配准差异与可确认的预测错误分开记录。

## 结果适用范围
当前实验输出为生成图像。图块来源的独立性按清单中的组标识层级描述；`unknown` 表示患者/切片层面的独立性尚未核实。像素指标和 DAB 颜色分量不能单独确定临床 HER2 状态。输入模糊分析对应本课程固定的图像处理条件。
'''
    (output/'实验结果自动汇总.md').write_text(text,encoding='utf-8')
    return text


@torch.inference_mode()
def infer_new_he(checkpoint,input_dir,output,scale_description='unknown'):
    model,meta=load_stain_model(checkpoint);size=meta['config']['size']
    files=sorted(p for p in Path(input_dir).rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES and p.is_file())
    if not files:raise FileNotFoundError('没有找到新的 H&E RGB 图块。请提供 PNG/JPEG/普通RGB TIFF。')
    output=Path(output);output.mkdir(parents=True,exist_ok=True);rows=[]
    for p in files:
        # 这里读普通图块；全切片需要 OpenSlide 等专门工具切块并保留尺度信息。
        im=read_rgb(p);a=np.asarray(im.resize((size,size),Image.Resampling.BILINEAR),dtype=np.float32)/255.
        pred=predict_rgb(model,a);fid=hashlib.sha256(str(p.resolve()).encode()).hexdigest()[:12]
        q=output/f'{p.stem}_{fid}_virtual_HER2_IHC.png'
        Image.fromarray(np.round(pred*255).astype('uint8')).save(q)
        rows.append(dict(input=p.name,input_sha256=sha_file(p),output=q.name,original_width=im.width,original_height=im.height,
            model_input_size=size,marker='HER2',reference_available=False,scale_description=scale_description,
            interpretation='model-generated image; unvalidated new-source inference'))
    pd.DataFrame(rows).to_csv(output/'inference_manifest.csv',index=False)
    write_json(output/'model_provenance.json',dict(checkpoint_sha256=sha_file(checkpoint),training_meta=meta,scale_description=scale_description))
    return pd.DataFrame(rows)


def zip_results(folder,output):
    folder=Path(folder).resolve();output=Path(output).resolve()
    with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(folder.rglob('*')):
            if p.is_file() and p.resolve()!=output:z.write(p,p.relative_to(folder))
    return output


## 1. 数据准备

按手册第5章从BCI官方入口获取并解压数据。将有权使用的数据添加到Kaggle Input。公开转存需单独确认来源许可。文件夹为HE/IHC时可自动识别；A/B目录或横向组合图需明确方向。第一次运行先读下方配置说明。

In [ ]:
BCI_ROOT=''  # 留空时在Input中寻找唯一可识别数据；也可填写确切目录
LAYOUT='auto_separate'  # auto_separate / a_to_b / b_to_a / combined
HE_SIDE=None  # combined时必须填left或right
METADATA_CSV=''  # 来源核实的pair_id+patient_id或slide_id；没有时留空
SEED=42
LIMITS={'train':1024,'val':256,'test':512}  # None表示全量
AUDIT_NAME='bci_audit_001'

In [ ]:
root=resolve_bci_root(BCI_ROOT,LAYOUT,HE_SIDE)
preview=discover_bci(root,LAYOUT,HE_SIDE)
print('根目录：',root,'总配对数：',len(preview))
print(preview.head().to_string(index=False))
he,ihc=read_pair(preview.iloc[0],root)
show_strip([np.asarray(he)/255.,np.asarray(ihc)/255.],['H&E: confirm purple/pink','HER2 IHC: confirm reference stain'])

## 2. 确认方向后建立清单

对应的组织形状应大体相符，目标图应来自实验染色。若图像左右或目录方向弄反，请修正配置后重新从第1步运行。全量数据先完成跨集合重复检查，课程子集在检查通过后产生。

In [ ]:
audit=fresh_run(work_root()/AUDIT_NAME)
manifest,card=prepare_bci(root,audit,LAYOUT,HE_SIDE,METADATA_CSV or None,SEED,LIMITS)
manifest=load_manifest(audit/'bci_manifest.csv')
print(json.dumps(card,ensure_ascii=False,indent=2))
print(manifest.groupby(['split','her2_label']).size())
verify_data_files(manifest,root,check_hashes=True)
print('配对、尺寸、划分与图像哈希检查通过。')

## 3. 观察多个配对

优先看组织边缘、较大的腔隙与核密集区域，判断配对是否存在位移。相邻区域或相邻切片的配准仍可有局部差异。

In [ ]:
for row in manifest.head(3).to_dict('records'):
    he,ihc=read_pair(row,root)
    he=he.resize((256,256),Image.Resampling.BILINEAR)
    ihc=ihc.resize((256,256),Image.Resampling.BILINEAR)
    print(row['pair_id'])
    show_strip([np.asarray(he)/255.,np.asarray(ihc)/255.],['H&E','Reference HER2-IHC'])
write_json(audit/'manual_review_template.json',{'checked_pair_ids':manifest.head(3).pair_id.tolist(),'direction_confirmed':'由学生核对后在学习记录中填写','registration_observations':'由学生根据图像填写'})

## 任务 P03

完成数据卡，记录实际数量、目录方向、group_level、原始像素尺寸与来源。`unknown`表示患者/切片独立性未核实，只能按当前图块划分范围解释结果。保留原始命名；文件名中的HER2来源类别不能作为每个像素的蛋白标注。保存本次Notebook版本，将输出文件作为04的Input。